**Document Analysis with GraphRAG & LLM**

__Building a Multi-Page Financial Knowledge Graph with LlamaParse, Groq, and Neo4j for Apple's 10Q.__

In [3]:
# Importing Necessary Libraries & Packages
import json
import os
from pathlib import Path
from typing import Any
from dotenv import load_dotenv
from groq import Groq
from neo4j import GraphDatabase
import os
import time
from pathlib import Path
from collections import defaultdict
import re
import nest_asyncio
import asyncio
nest_asyncio.apply()
# Ensure a global loop is set and stays open
try:
    asyncio.get_event_loop()
except RuntimeError:
    asyncio.set_event_loop(asyncio.new_event_loop())

In [4]:
# Load environment variables from .env
load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
#OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LLAMA_API_KEY = os.getenv("LLAMA_API_KEY")

# Verifying all the credentials are loaded correctly
if not all([NEO4J_URI,NEO4J_USERNAME,NEO4J_PASSWORD,GROQ_API_KEY,LLAMA_API_KEY]):
    raise ValueError("Missing credentials in .env file.")

#print(f"Neo4j URI: {NEO4J_URI}")
#print(f"GROQ API Key: {GROQ_API_KEY}")


**The Data Ingestion to Knowled-Graph Pipeline:**

***Ingestion Pipeline - LlamaParse (Document Extraction) -> Chunking data for LLM -> LLM (Groq) based Entity / Relation Extraction -> Neo4j Graph Generation***

In [255]:
# Using LLamaParse for Data Extraction and Data Ingestion for LLM
from llama_parse import LlamaParse 
parser = LlamaParse(
    api_key = LLAMA_API_KEY,
    result_type = "markdown"
)


In [ ]:

# Function to load all PDF's from the directory and extract text using llama_parse.

def all_fin_pdfs(pdf_directory):
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("*.pdf"))  # Finds all PDFs in the folder
    
    all_docs = []

    print(f"📂 Found {len(pdf_files)} PDF files. Processing one by one...")

    for pdf_path in pdf_files:
        file_name = pdf_path.name  # e.g., 'Apple_10Q.pdf'
        print(f"📄 Parsing: {file_name}...")
        
        try:
            # Parse this specific file ONLY
            # This returns a list of Document objects (one per page usually)
            single_file_pages = parser.load_data(str(pdf_path))
            
            # STAMP the metadata on every single page from this file
            for page in single_file_pages:
                page.metadata["source"] = file_name
                page.metadata["file_path"] = str(pdf_path)
                # You can even add a page label if you want
                # page.metadata["doc_id"] = f"{file_name}_page"
            
            all_docs.extend(single_file_pages)
            
        except Exception as e:
            print(f"❌ Failed to parse {file_name}: {e}")

    print(f"✅ Finished! Total pages collected: {len(all_docs)}")
    return all_docs

all_docs = all_fin_pdfs("../data/financial_pdf")

In [ ]:
def group_docs_by_source(all_docs):
    """
    Groups individual pages into consolidated documents based on the 'source' filename.
    """
    grouped_content = defaultdict(list)
    
    for doc in all_docs:
        # Retrieve the stamped source name
        source = doc.metadata.get("source", "unknown_document.pdf")
        # Append the text of the page to the list for that specific source
        grouped_content[source].append(doc.text)

    # Convert the lists of pages into single strings
    combined_docs = []
    for source, pages in grouped_content.items():
        combined_text = "\n\n--- PAGE BREAK ---\n\n".join(pages)
        
        combined_docs.append({
            "source": source,
            "text": combined_text,
            "fiscal_year": "FY2025"  # You can make this dynamic if needed
        })
        
        print(f"✅ Combined {len(pages)} pages for: {source}")

    return combined_docs

# --- EXECUTION ---
# 1. Group the pages
sample_docs = group_docs_by_source(all_docs)

✅ Combined 28 pages for: 10Q-Q1-2026-as-filed.pdf


In [ ]:
# Based on the document type differnt prompt input is generated.

def detect_document_type(doc_source: str) -> str:
    """ Detect document type based on filename"""
    source_name = doc_source.lower()
    print(f"{source_name}")

    if "mortgage" in source_name or "loan" in source_name:
        return "mortgage"
    elif "10-q" in source_name or "10q" in source_name:
        return "10Q"
    else:
        return "financial"
  
def get_extraction_prompt(doc_type: str, doc_text: str, doc_source: str, fiscal_year: str) -> str:
    """ 
    Generating document type specific prompt
    
    Args:
        doc_type: '10Q', 'Annual Report', 'Mortgage statement'.
        doc_text: The markdown from Llama_parse to extract info from.
        doc_source: file_name / source location.
        fiscal_year: year or perid - Ex: 'FY2025','2024-2025'.
    """
    if doc_type == "mortgage":
        # Mortgage document - info extraction prompt
        return  f"""You are analyzing a mortgage document template. Extract ONLY the basic loan structure details.
        Return ONLY valid JSON, no explanation.
        Document: {doc_source}
        Fiscal Period: {fiscal_year}
 
        Text:
        {doc_text[:2000]}
        
        Return EXACTLY this JSON structure:
        {{
        "entities": [
            {{"name": "Loan Amount", "type": "Metric", "value": "extracted principal amount if present"}},
            {{"name": "Interest Rate", "type": "Metric", "value": "extracted rate if present"}},
            {{"name": "Loan Term", "type": "Metric", "value": "extracted term in years if present"}},
            {{"name": "Lender Name", "type": "Company", "value": "lender name if present"}}
        ],
        "relations": [
            {{"source": "Lender Name", "relation": "PROVIDED_LOAN", "target": "Loan Amount", "metadata": {{"term_years": "value if found"}}}}
        ]
        }}
        """
    elif doc_type == "10Q":
        # 10-Q document -info extraction prompt
        return f"""You are a Financial Data Engineer. Extract a detailed Knowledge Graph from this SEC 10-Q filing.
        
        SOURCE: {doc_source}
        PERIOD: {fiscal_year}

        TEXT:
        {doc_text}

        EXTRACT THE FOLLOWING ENTITIES:
        1. COMPANY: Full legal name, ticker, and headquarters.
        2. LEADERSHIP: All C-suite members (CEO, CFO, etc.) mentioned.
        3. METRICS: Revenue, Net Income, EPS, and Debt (include units like 'USD Billions').
        4. SEGMENTS: Business units (e.g., iPhone, Services, Wearables).
        5. RISKS: Comprehensive risk factors. Capture the full descriptive title of the risk.
        6. SUPPLIERS/PARTNERS: Any key entities in the supply chain.

        EXTRACT RELATIONSHIPS:
        - (Company)-[:REPORTED]->(Metric)
        - (Company)-[:LED_BY]->(Person)
        - (Company)-[:OPERATES_SEGMENT]->(Segment)
        - (Company)-[:FACES_RISK]->(Risk)
        - (Segment)-[:CONTRIBUTED_REVENUE]->(Metric)

        RETURN ONLY VALID JSON:
        {{
        "entities": [
            {{"name": "Apple Inc", "type": "Company", "properties": {{"ticker": "AAPL", "hq": "Cupertino"}}}}
        ],
        "relations": [
            {{"source": "Apple Inc", "relation": "REPORTED", "target": "Revenue", "metadata": {{"value": "383B", "year": "2025"}}}}
        ]
        }}
        """

In [ ]:
# Extracting info from chunked content, based on the document type specific prompt.

def extract_entities_with_groq(
    doc_text: str, 
    doc_source: str,
    fiscal_year: str = "FY2025"
) -> dict[str, Any]:
    """Extract entities using Groq with robust JSON handling"""
    print(f"{doc_source},{fiscal_year}")
    
    # Initialize client (Best practice: do this outside the function if calling in a loop)
    client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    
    doc_type = detect_document_type(doc_source)
    print(f"{doc_type}")

    prompt = get_extraction_prompt(doc_type, doc_text, doc_source, fiscal_year)

    if not prompt or not isinstance(prompt, str) or len(prompt.strip()) == 0:
        print(f"  ✗ Invalid prompt for {doc_source}")
        return {"entities": [], "relations": []}
    
    try:
        print(f"  -> Extracting {doc_type.upper()} via Groq...")
        
        # Use JSON Mode: This forces Groq to output a valid JSON object
        response = client.chat.completions.create(
            # Try this EXACT string first:
            model="llama-3.3-70b-versatile", 
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            response_format={"type": "json_object"} 
        )

        # Groq's SDK uses attribute access: .choices[0].message.content
        response_text = response.choices[0].message.content.strip()

        # Find the first '{' and the last '}'
        match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if match:
            response_text = match.group(0)
        
        # Parse JSON
        extracted = json.loads(response_text)
        
        # Validation of keys to prevent downstream errors in Neo4j
        if "entities" not in extracted: extracted["entities"] = []
        if "relations" not in extracted: extracted["relations"] = []

        print(f"  ✓ Success: {len(extracted['entities'])} entities, {len(extracted['relations'])} relations.")
        return extracted
    
    except json.JSONDecodeError as e:
        print(f"  ✗ JSON Decoder Error on {doc_source}: {e}")
        # Log the snippet of text that caused the error for debugging
        print(f"    Raw snippet: {response_text[:100]}...")
        return {"entities": [], "relations": []}
    except Exception as e:
        print(f"  ✗ Groq API/Runtime Error: {e}")
        return {"entities": [], "relations": []}

In [ ]:
# Neo4j Ingestion Class for Creating Nodes & Connection input for Knowledge Graph

class Neo4jIngestor:
    """ Handles Neo4j schema creation and write operation for financial / mortgage documents """

    def __init__(self, uri, username, password):
        self.driver = GraphDatabase.driver(uri, auth=(username, password))
        self.setup_schema()

    # Automatic Connection Handling ---
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.driver.close()
        print("✔ Neo4j Driver closed automatically.")

    # Schema Setup
    def setup_schema(self):
        """Initializes constraints to prevent duplicate nodes."""
        with self.driver.session() as session:
            # This is the most important constraint for your GraphRAG
            session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE")
            session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (c:Company) REQUIRE c.name IS UNIQUE")
            print("✓ Neo4j schema constraints initialized.")

    # Public method to ingest extracted data
    def ingest_extracted_data(self, extracted: dict, doc_source: str):
        """This is the method the pipeline calls to start the work."""
        entities = extracted.get("entities", [])
        relations = extracted.get("relations", [])

        with self.driver.session() as session:
            # Create all entities first
            for entity in entities:
                self._create_entity_node(session, entity, doc_source)

            # Create relationships once nodes exist
            for relation in relations:
                self._create_relationship(session, relation, doc_source)

    # Internal Helpers - Node Creation
    def _create_entity_node(self, session, entity: dict, doc_source: str):
        name = entity.get("name", "").strip()
        # Clean the label: No spaces allowed in Neo4j labels!
        entity_type = entity.get("type", "Entity").strip().replace(" ", "_")
        
        if not name: return

        # Prepare the properties as a safe dictionary
        properties = {
            "name": name,
            "source_file": doc_source,
            "last_updated": "datetime()"
        }

        # Merge in extra properties from Groq
        if "properties" in entity:
            properties.update(entity["properties"])
        
        # Construct the query
        # We use an f-string ONLY for the Label (the part Neo4j won't parameterize)
        # We use $name and $props for everything else to keep it safe.
        query = f"""
        MERGE (e:Entity {{name: $name}})
        SET e += $props
        SET e:{entity_type}
        RETURN e
        """
        
        try:
            session.run(query, name=name, props=properties)
        except Exception as e:
            # If 'Apple Inc' still conflicts, it's likely a leftover constraint 
            # from a previous run with a different label.
            print(f"⚠️ Conflict on {name}: {e}")
        
    def _create_relationship(self, session, relation: dict, doc_source: str):
            source = relation.get("source", "").strip()
            target = relation.get("target", "").strip()
            rel_type = relation.get("relation", "RELATED_TO").strip()
            metadata = relation.get("metadata", {})

            if not source or not target: return

            query = f"""
            MATCH (a:Entity {{name: $source}})
            MATCH (b:Entity {{name: $target}})
            MERGE (a)-[r:{rel_type}]->(b)
            SET r += $metadata, r.source_file = $source_file, r.last_updated = datetime()
            """
            session.run(query, source=source, target=target, metadata=metadata, source_file=doc_source)

    # Reporting
    
    def verify_ingestion(self):
        """Show graph stats after processing."""
        with self.driver.session() as session:
            nodes = session.run("MATCH (n) RETURN count(n) as count").single()["count"]
            rels = session.run("MATCH ()-[r]->() RETURN count(r) as count").single()["count"]

            node_types = session.run("""
                MATCH (n) 
                UNWIND labels(n) AS label
                RETURN label, count(n) AS count
                ORDER BY count DESC
            """).data()
            
            print(f"\n📊 --- Neo4j Graph Stats ---")
            print(f"  Total Nodes: {nodes}")
            print(f"  Total Relationships: {rels}")
            print(f"  Nodes by Type:")
            for row in node_types:
                print(f"    - {row['label']}: {row['count']}")
            print(f"---------------------------\n")

In [ ]:
# Main pipepline - Execution flow:

def main_pipeline(combined_docs):
    """
    The Orchestrator: 
    1. Initializes Neo4j (via Context Manager)
    2. Chunks 50-page texts
    3. Triggers Groq Extraction
    4. Ingests to Neo4j
    """
    
    # Initialize using 'with' to handle the connection lifecycle automatically
    print("🔗 Connecting to Neo4j...")
    
    try:
        # Use 'with' so the driver closes itself even if an error occurs
        with Neo4jIngestor(
            uri=os.getenv("NEO4J_URI"),
            username=os.getenv("NEO4J_USERNAME"),
            password=os.getenv("NEO4J_PASSWORD")
        ) as db:

            for doc in combined_docs:
                source_file = doc['source']
                fiscal_year = doc['fiscal_year']
                full_text = doc['text']
                print(f"{source_file}")
                
                print(f"\n🚀 STARTING DOCUMENT: {source_file} ({len(full_text)} characters)")
                
                # Chunking Strategy
                chunk_size = 12000 
                total_chunks = (len(full_text) // chunk_size) + 1
                
                for i in range(0, len(full_text), chunk_size):
                    current_chunk_num = (i // chunk_size) + 1
                    print(f"  → Processing Chunk {current_chunk_num}/{total_chunks}...")
                    
                    chunk_text = full_text[i : i + chunk_size]
                    
                    try:
                        # Call Groq Extraction
                        extracted_json = extract_entities_with_groq(
                            chunk_text, 
                            source_file, 
                            fiscal_year
                        )
                        
                        # Ingest into Neo4j using the bridge method
                        if extracted_json.get("entities") or extracted_json.get("relations"):
                            db.ingest_extracted_data(extracted_json, source_file)
                            print(f"    ✅ Ingested {len(extracted_json.get('entities', []))} entities.")
                        else:
                            print(f"    ⚠️ Chunk {current_chunk_num} yielded no data.")
                            
                    except Exception as chunk_error:
                        print(f"    ❌ Error in Chunk {current_chunk_num}: {chunk_error}")
                    
                    # Rate Limit Protection
                    time.sleep(2) 

                print(f"🏁 Finished processing: {source_file}")

            # Final verification report
            db.verify_ingestion()

    except Exception as e:
        print(f"🚨 Pipeline failed: {e}")

# Execution Main Function
if __name__ == "__main__":
    if 'sample_docs' in locals() or 'sample_docs' in globals():
        main_pipeline(sample_docs)
    else:
        print("Error: 'sample_docs' not found. Ensure LlamaParse code ran correctly.")

**Retrieval-Augmented Generation (RAG) with Graph Databases:**

***Query Analysis - LLM Turns User Query to Cypher Query -> Graph Traversal - Neo4j Executes Cypher Query - Finds Specific Nodes & Connections -> LLM Generates Ouput based on Graph Data***

In [ ]:
# Graph Assistant - Querying the Graph using Groq LLM for Text-to-Cypher and Final Answer Generation

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
db = Neo4jIngestor(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD)

def ask_graph_assistant(user_query, ingestor, groq_client):
    # Text-to-Cypher
    cypher_prompt = f"""
    You are a Neo4j Expert. Convert this question into Cypher.
    
    SCHEMA:
    - Labels: (Entity), (Company), (Risk), (Metric)
    - Relationships: 
        (Company)-[:FACES_RISK]->(Risk)
        (Company)-[:REPORTED]->(Metric)
        (Company)-[:LED_BY]->(Entity)
    
    INSTRUCTIONS:
    - Use 'toLower(n.name) CONTAINS toLower("apple")' for matching.
    - Return ONLY the raw Cypher query. No markdown.
    
    QUESTION: {user_query}
    """
    
    cypher_res = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": cypher_prompt}]
    )

    # Strip markdown backticks and 'cypher' tags
    raw_query = cypher_res.choices[0].message.content.strip()
    clean_query = raw_query.replace("```cypher", "").replace("```", "").strip()
    
    print(f"🕵️ Searching Graph for: {clean_query}")

    # Now execute the CLEANED query
    with ingestor.driver.session() as session:
        results = session.run(clean_query).data()

    # Final Summary
    final_prompt = f"Using this data: {json.dumps(results)}, answer: {user_query}"
    final_res = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": final_prompt}]
    )
    
    return final_res.choices[0].message.content

# TEST
print(ask_graph_assistant("What are the risks for Apple?", db, client))

✓ Neo4j schema constraints initialized.
🕵️ Searching Graph for: MATCH (c:Company)-[:FACES_RISK]->(r:Risk) WHERE toLower(c.name) CONTAINS toLower("apple") RETURN r
The data provided lists various risks associated with a company, which appears to be Apple based on the context and the reference to a Form 10-K and Form 10-Q, which are typical SEC filings for publicly traded companies in the United States. The risks can be categorized into several areas:

1. **Economic Risks**:
   - Economic Downturn: Risk of economic downturn affecting sales.
   - Global Economic Downturn: Risks related to global economic downturn that may impact demand for products.
   - Global Economic Uncertainty: Risk of global economic downturn affecting sales and revenue.
   - Macroeconomic Conditions: Including inflation, interest rates, component pricing, and currency fluctuations.

2. **Operational Risks**:
   - Supply Chain Disruptions: Risk of supply chain disruptions affecting production and delivery.
   - Supp